In [ ]:
# Environment setup
import subprocess
subprocess.run([
    "pip", "install",
    "tensorflow==2.12.0",
    "numpy", "pandas", "matplotlib",
    "seaborn", "pillow", "scikit-learn"
], check=True)

# 02 Experiments: 72-Run Simulation
Trains all **72 model configurations** (4 architectures × 6 optimizers × 3 learning rates) and saves every artifact to `results/`.

---
## Section 0 : Imports and Configuration

In [ ]:
import os
import pathlib
import random
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Global random seed 
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

# Paths 
PROJECT_ROOT = pathlib.Path('..').resolve()
PROCESSED_DIR = PROJECT_ROOT / 'dataset' / 'processed'
RESULTS_DIR   = PROJECT_ROOT / 'results'

METRICS_DIR      = RESULTS_DIR / 'metrics'
CURVES_ACC_DIR   = RESULTS_DIR / 'curves' / 'accuracy_loss'
CURVES_ROC_DIR   = RESULTS_DIR / 'curves' / 'auc_roc'
CM_DIR           = RESULTS_DIR / 'confusion_matrices'
SUMMARY_DIR      = RESULTS_DIR / 'summary'

for d in [METRICS_DIR, CURVES_ACC_DIR, CURVES_ROC_DIR, CM_DIR, SUMMARY_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Experiment matrix 
ARCHITECTURES  = ['VGG19', 'EfficientNetB3', 'ResNet50', 'DenseNet121']
OPTIMIZERS     = ['Adam', 'Adagrad', 'Adamax', 'AdaDelta', 'SGD', 'RMSProp']
LEARNING_RATES = [1e-4, 1e-5, 1e-6]

# Fixed training hyperparameters 
EPOCHS     = 50
BATCH_SIZE = 32
INPUT_SIZE = (256, 256)

total_runs = len(ARCHITECTURES) * len(OPTIMIZERS) * len(LEARNING_RATES)
print(f"Experiment matrix: {len(ARCHITECTURES)} arch × {len(OPTIMIZERS)} opt × {len(LEARNING_RATES)} LR = {total_runs} runs")

---
## Section 0.5: Function Definitions

### Model Builder

In [ ]:
import tensorflow as tf
from tensorflow.keras import Model
from tensorflow.keras.applications import VGG19, EfficientNetB3, ResNet50, DenseNet121
from tensorflow.keras.layers import (
    GlobalAveragePooling2D, Dense, Dropout, BatchNormalization,
)
from tensorflow.keras.metrics import Precision, Recall, AUC
from tensorflow.keras.optimizers import (
    Adam, Adagrad, Adamax, Adadelta, SGD, RMSprop,
)

# Architecture registry 

_ARCHITECTURE_REGISTRY = {
    "VGG19":           VGG19,
    "EfficientNetB3":  EfficientNetB3,
    "ResNet50":        ResNet50,
    "DenseNet121":     DenseNet121,
}

# Optimizer registry 
def _make_optimizer(name: str, learning_rate: float):
    """Instantiate an optimizer by name with the given learning rate."""
    registry = {
        "Adam":     lambda lr: Adam(learning_rate=lr),
        "Adagrad":  lambda lr: Adagrad(learning_rate=lr),
        "Adamax":   lambda lr: Adamax(learning_rate=lr),
        "AdaDelta": lambda lr: Adadelta(learning_rate=lr),
        "SGD":      lambda lr: SGD(learning_rate=lr, momentum=0.9),
        "RMSProp":  lambda lr: RMSprop(learning_rate=lr),
    }
    if name not in registry:
        raise ValueError(
            f"Unknown optimizer '{name}'. "
            f"Choose from: {sorted(registry.keys())}"
        )
    return registry[name](learning_rate)


# Model builder 
def build_model(
    architecture_name: str,
    optimizer_name: str,
    learning_rate: float,
    input_shape: tuple = (256, 256, 3),
) -> Model:
    if architecture_name not in _ARCHITECTURE_REGISTRY:
        raise ValueError(
            f"Unknown architecture '{architecture_name}'. "
            f"Choose from: {sorted(_ARCHITECTURE_REGISTRY.keys())}"
        )

    # Base model (frozen ImageNet weights) 
    base_cls = _ARCHITECTURE_REGISTRY[architecture_name]
    base_model = base_cls(
        include_top=False,
        weights="imagenet",
        input_shape=input_shape,
    )
    base_model.trainable = False

    # Custom classification head 
    x = base_model.output
    x = GlobalAveragePooling2D()(x)

    x = Dense(256, activation="relu")(x)
    x = BatchNormalization()(x)
    x = Dropout(0.6)(x)

    x = Dense(128, activation="relu")(x)
    x = BatchNormalization()(x)
    x = Dropout(0.4)(x)

    x = Dense(64, activation="relu")(x)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)

    output = Dense(1, activation="sigmoid")(x)

    model = Model(inputs=base_model.input, outputs=output)

    # Compile 
    optimizer = _make_optimizer(optimizer_name, learning_rate)
    model.compile(
        optimizer=optimizer,
        loss="binary_crossentropy",
        metrics=[
            "accuracy",
            Precision(name="precision"),
            Recall(name="recall"),
            AUC(name="auc"),
        ],
    )

    return model

### Training

In [ ]:
import pathlib

import tensorflow as tf
from tensorflow.keras.callbacks import (
    CSVLogger,
    EarlyStopping,
    ModelCheckpoint,
    ReduceLROnPlateau,
)


def train_model(
    model: tf.keras.Model,
    train_generator,
    val_generator,
    test_generator,
    architecture: str,
    optimizer_name: str,
    learning_rate: float,
    results_base_dir,
    epochs: int = 50,
) -> dict:
    results_base_dir = pathlib.Path(results_base_dir)

    # Step 1 - Setup 
    lr_str   = f"{learning_rate:.0e}"                    # e.g. '1e-04'
    run_name = f"{architecture}_{optimizer_name}_LR{lr_str}"
    run_dir  = results_base_dir / run_name
    run_dir.mkdir(parents=True, exist_ok=True)

    print("=" * 60)
    print(f"TRAINING: {run_name}")
    print(f"Architecture : {architecture}")
    print(f"Optimizer    : {optimizer_name}")
    print(f"Learning Rate: {learning_rate}")
    print("=" * 60)

    # Step 2 - Callbacks 
    callbacks = [
        ModelCheckpoint(
            filepath=str(run_dir / "best_model.keras"),
            monitor="val_auc",
            mode="max",
            save_best_only=True,
            verbose=0,
        ),
        EarlyStopping(
            monitor="val_auc",
            patience=10,
            mode="max",
            restore_best_weights=True,
            min_delta=0.001,
            verbose=1,
        ),
        CSVLogger(
            filename=str(run_dir / "training_log.csv"),
            append=False,
        ),
        ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=5,
            min_lr=1e-7,
            verbose=1,
        ),
    ]

    # Step 3 - Training
    history = model.fit(
        train_generator,
        validation_data=val_generator,
        epochs=epochs,
        callbacks=callbacks,
        verbose=1,
    )

    actual_epochs  = len(history.history["loss"])
    best_val_auc   = max(history.history["val_auc"])
    print(f"\nTraining complete - epochs run: {actual_epochs} / {epochs}  |  best val_auc: {best_val_auc:.4f}")

    # Step 4 - Evaluation and Visualization 
    metrics = evaluate_model(
        model=model,
        test_generator=test_generator,
        run_name=run_name,
        output_dir=str(run_dir),
    )

    # saves {run_name}_acc_loss.png into run_dir
    plot_accuracy_loss(
        history=history,
        run_name=run_name,
        output_dir=str(run_dir),
    )

    # saves {run_name}_auc_roc.png into run_dir
    plot_auc_roc(
        model=model,
        test_generator=test_generator,
        run_name=run_name,
        output_dir=str(run_dir),
    )

    metrics["epochs_run"] = actual_epochs

    # Step 5 - Cleanup 
    tf.keras.backend.clear_session()

    print(f"\nCOMPLETED: {run_name}")
    print("=" * 60)

    return metrics

### Evaluation

In [ ]:
import os
import csv
import math

import numpy as np
from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    accuracy_score,
)


def evaluate_model(
    model,
    test_generator,
    run_name: str,
    output_dir: str,
) -> dict:
    # Prediction
    # Reset generator so prediction starts at the first batch.
    test_generator.reset()
    y_prob = model.predict(test_generator, verbose=0).ravel()
    y_pred = (y_prob >= 0.5).astype(int)
    y_true = test_generator.classes

    # Scalar Metrics
    accuracy  = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall    = recall_score(y_true, y_pred, zero_division=0)
    f1        = f1_score(y_true, y_pred, zero_division=0)
    auc       = roc_auc_score(y_true, y_prob)

    # Specificity: TN / (TN + FP) 
    cm = confusion_matrix(y_true, y_pred)
    tn, fp = cm[0, 0], cm[0, 1]
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    # Parse run_name into components 
    # Expected format: {Architecture}_{Optimizer}_LR{lr}  e.g. ResNet50_Adam_LR1e-4
    parts = run_name.split("_")
    architecture   = parts[0] if len(parts) > 0 else run_name
    optimizer_name = parts[1] if len(parts) > 1 else ""
    learning_rate  = parts[2].replace("LR", "") if len(parts) > 2 else ""

    # Confusion matrix plot 
    cm_dir = os.path.join(output_dir, "confusion_matrices")
    os.makedirs(cm_dir, exist_ok=True)
    plot_confusion_matrix(
        cm=cm,
        run_name=run_name,
        class_names=list(test_generator.class_indices.keys()),
        output_dir=cm_dir,
    )

    # Save metrics CSV (one row per run) 
    metrics_dir = os.path.join(output_dir, "metrics")
    os.makedirs(metrics_dir, exist_ok=True)
    csv_path = os.path.join(metrics_dir, f"{run_name}.csv")

    metrics = {
        "run_name":     run_name,
        "architecture": architecture,
        "optimizer":    optimizer_name,
        "learning_rate": learning_rate,
        "accuracy":     round(accuracy,    4),
        "precision":    round(precision,   4),
        "recall":       round(recall,      4),
        "specificity":  round(specificity, 4),
        "f1":           round(f1,          4),
        "auc":          round(auc,         4),
    }

    with open(csv_path, "w", newline="") as fh:
        writer = csv.DictWriter(fh, fieldnames=list(metrics.keys()))
        writer.writeheader()
        writer.writerow(metrics)

    print(
        f"[evaluate] {run_name} | "
        f"acc={accuracy:.4f}  prec={precision:.4f}  rec={recall:.4f}  "
        f"spec={specificity:.4f}  f1={f1:.4f}  auc={auc:.4f}"
    )

    return metrics

### Visualization

In [ ]:
import os

import numpy as np
import matplotlib
matplotlib.use("Agg")  # non-interactive backend - safe for headless runs
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, auc as sklearn_auc

# Shared style Settings

_STYLE        = "seaborn-v0_8-whitegrid"
_FIG_DPI      = 150
_CANCER_COLOR = "#C0392B"   # red - cancer class
_NORMAL_COLOR = "#2980B9"   # blue - no_cancer class


# Accuracy / Loss curves 

def plot_accuracy_loss(history, run_name: str, output_dir: str) -> None:
    os.makedirs(output_dir, exist_ok=True)

    h = history.history if hasattr(history, "history") else history
    epochs = range(1, len(h["accuracy"]) + 1)

    with plt.style.context(_STYLE):
        fig, (ax_acc, ax_loss) = plt.subplots(1, 2, figsize=(12, 4))
        fig.suptitle(run_name, fontsize=11, fontweight="bold")

        # Accuracy subplot
        ax_acc.plot(epochs, h["accuracy"],     label="Train Accuracy", linewidth=1.8)
        ax_acc.plot(epochs, h["val_accuracy"], label="Val Accuracy",   linewidth=1.8, linestyle="--")
        ax_acc.set_title("Accuracy")
        ax_acc.set_xlabel("Epoch")
        ax_acc.set_ylabel("Accuracy")
        ax_acc.legend()
        ax_acc.set_ylim(0, 1)

        # Loss subplot
        ax_loss.plot(epochs, h["loss"],     label="Train Loss", linewidth=1.8)
        ax_loss.plot(epochs, h["val_loss"], label="Val Loss",   linewidth=1.8, linestyle="--")
        ax_loss.set_title("Loss")
        ax_loss.set_xlabel("Epoch")
        ax_loss.set_ylabel("Binary Cross-Entropy")
        ax_loss.legend()

        fig.tight_layout()
        out_path = os.path.join(output_dir, f"{run_name}_acc_loss.png")
        fig.savefig(out_path, dpi=_FIG_DPI, bbox_inches="tight")
        plt.close(fig)

    print(f"[visualize] Accuracy/Loss curve saved -> {out_path}")


# AUC-ROC curve 

def plot_auc_roc(model, test_generator, run_name: str, output_dir: str) -> None:
    os.makedirs(output_dir, exist_ok=True)

    test_generator.reset()
    y_prob = model.predict(test_generator, verbose=0).ravel()
    y_true = test_generator.classes

    fpr, tpr, _ = roc_curve(y_true, y_prob)
    roc_auc     = sklearn_auc(fpr, tpr)

    with plt.style.context(_STYLE):
        fig, ax = plt.subplots(figsize=(6, 5))
        ax.plot(fpr, tpr, color=_CANCER_COLOR, linewidth=2,
                label=f"AUC = {roc_auc:.4f}")
        ax.plot([0, 1], [0, 1], color="grey", linestyle="--", linewidth=1,
                label="Random Classifier")
        ax.set_xlabel("False Positive Rate")
        ax.set_ylabel("True Positive Rate")
        ax.set_title(f"ROC Curve - {run_name}", fontweight="bold")
        ax.legend(loc="lower right")
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1.02)

        fig.tight_layout()
        out_path = os.path.join(output_dir, f"{run_name}_auc_roc.png")
        fig.savefig(out_path, dpi=_FIG_DPI, bbox_inches="tight")
        plt.close(fig)

    print(f"[visualize] AUC-ROC curve saved -> {out_path}")


# Confusion matrix 

def plot_confusion_matrix(
    cm: np.ndarray,
    run_name: str,
    class_names: list,
    output_dir: str,
) -> None:
    os.makedirs(output_dir, exist_ok=True)

    with plt.style.context(_STYLE):
        fig, ax = plt.subplots(figsize=(5, 4))

        # Annotate each cell with count and row percentage
        total_per_row = cm.sum(axis=1, keepdims=True)
        pct = cm / total_per_row.clip(min=1) * 100
        annot = np.array(
            [[f"{cm[i, j]}\n({pct[i, j]:.1f}%)" for j in range(cm.shape[1])]
             for i in range(cm.shape[0])]
        )

        sns.heatmap(
            cm,
            annot=annot,
            fmt="",
            cmap="Blues",
            xticklabels=class_names,
            yticklabels=class_names,
            linewidths=0.5,
            ax=ax,
            cbar_kws={"shrink": 0.75},
        )
        ax.set_xlabel("Predicted Label", fontsize=10)
        ax.set_ylabel("True Label", fontsize=10)
        ax.set_title(f"Confusion Matrix - {run_name}", fontweight="bold", fontsize=11)

        fig.tight_layout()
        out_path = os.path.join(output_dir, f"{run_name}.png")
        fig.savefig(out_path, dpi=_FIG_DPI, bbox_inches="tight")
        plt.close(fig)

    print(f"[visualize] Confusion matrix saved -> {out_path}")

---
## Section 1 : Shared Data Pipeline

In [ ]:
# Generator definitions (constructed ONCE here, then reused across all 72 runs) 

train_datagen = ImageDataGenerator(
    samplewise_center=True,
    samplewise_std_normalization=True,
    rotation_range=30,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.3,
    zoom_range=0.30,
    horizontal_flip=True,
    fill_mode='nearest',
)

eval_datagen = ImageDataGenerator(
    samplewise_center=True,
    samplewise_std_normalization=True,
)

train_generator = train_datagen.flow_from_directory(
    PROCESSED_DIR / 'train',
    target_size=INPUT_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=True,
    seed=RANDOM_SEED,
)

val_generator = train_datagen.flow_from_directory(
    PROCESSED_DIR / 'val',
    target_size=INPUT_SIZE,
    batch_size=1,
    class_mode='binary',
    shuffle=True,
    seed=RANDOM_SEED,
)

test_generator = eval_datagen.flow_from_directory(
    PROCESSED_DIR / 'test',
    target_size=INPUT_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False,
)

CLASS_NAMES = list(train_generator.class_indices.keys())
print(f"Train : {train_generator.samples} images")
print(f"Val   : {val_generator.samples} images")
print(f"Test  : {test_generator.samples} images")
print(f"Classes: {train_generator.class_indices}")

---
## Section 2 : Pipeline Configuration Reference

Training was run via `src/run_all.py`. This section displays the exact pipeline configuration
and experiment matrix that was used during training, for documentation and reproducibility.
No model training is performed here.

**Run naming convention:** `{Architecture}_{Optimizer}_LR{lr}`  
e.g., `ResNet50_Adam_LR1e-04`

**Output files produced per run** (written to `results/{run_name}/`):
- `{run_name}_acc_loss.png` : training and validation accuracy/loss curves
- `{run_name}_auc_roc.png` : AUC-ROC curve on the test set
- `confusion_matrices/{run_name}.png` : confusion matrix on the test set

In [ ]:
import itertools

# Display the full experiment matrix 
print("=" * 60)
print("EXPERIMENT MATRIX (as executed by src/run_all.py)")
print("=" * 60)
print(f"  Architectures  : {ARCHITECTURES}")
print(f"  Optimizers     : {OPTIMIZERS}")
print(f"  Learning rates : {LEARNING_RATES}")
print(f"  Total runs     : {total_runs}")
print()

# Display fixed hyperparameters 
print("FIXED HYPERPARAMETERS")
print("-" * 40)
print(f"  Random seed    : {RANDOM_SEED}")
print(f"  Epochs (max)   : {EPOCHS}")
print(f"  Batch size     : {BATCH_SIZE}")
print(f"  Input size     : {INPUT_SIZE}")
print(f"  Dropout        : 0.6 / 0.4 / 0.3  (Dense layers 256/128/64)")
print(f"  Loss           : binary_crossentropy")
print(f"  Monitor metric : val_auc  (EarlyStopping, patience=10)")
print()

# Display training augmentation settings 
print("TRAINING AUGMENTATION (train generator only)")
print("-" * 40)
aug_params = {
    "samplewise_center":            True,
    "samplewise_std_normalization": True,
    "rotation_range":               30,
    "width_shift_range":            0.15,
    "height_shift_range":           0.15,
    "shear_range":                  0.3,
    "zoom_range":                   0.30,
    "horizontal_flip":              True,
    "fill_mode":                    "nearest",
}
for k, v in aug_params.items():
    print(f"  {k:<36}: {v}")
print()

# List all 72 run names in order 
print("ALL 72 RUN NAMES (execution order)")
print("-" * 40)
run_names = [
    f"{arch}_{opt}_LR{lr:.0e}"
    for arch, opt, lr in itertools.product(ARCHITECTURES, OPTIMIZERS, LEARNING_RATES)
]
for i, name in enumerate(run_names, start=1):
    print(f"  {i:02d}. {name}")

---
## Section 3 : Aggregate Results

Load all per-run CSVs from `results/metrics/`, compile into one master DataFrame, and save as `results/summary/all_runs_summary.csv`. 

In [ ]:
MASTER_CSV = SUMMARY_DIR / 'master_results.csv'

if not MASTER_CSV.exists():
    raise FileNotFoundError(
        f"Master results file not found: {MASTER_CSV}\n"
        "Run 'python src/run_all.py' from the project root to generate it."
    )

master_df = pd.read_csv(MASTER_CSV)

print(f"Master results loaded from: {MASTER_CSV}")
print(f"Total rows: {len(master_df)}  (expected 72)")
print()

pd.set_option('display.max_rows', 90)
pd.set_option('display.float_format', '{:.4f}'.format)

display(master_df.sort_values('auc', ascending=False).reset_index(drop=True))